# Phase 2 — Baseline EEG Models

This notebook trains simple baseline models to evaluate signal quality and establish reference performance on the I-CARE EEG dataset.

### Objectives
- Load preprocessed EEG segments
- Build and train simple classifiers (CNN, MLP)
- Evaluate baseline accuracy, ROC-AUC, and F1-score


## Parameters (Papermill)

In [2]:
# Parameters (Papermill injects values here)
preprocessed_path = '../data/icare/preprocessed'
epochs = 10
batch_size = 32
model_type = 'EEGNet'  # options: 'EEGNet', 'MLP'


## Load data

In [3]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path

# Simple loader for .npy segment arrays
def load_data(path):
    X, y = [], []
    for file in Path(path).glob('*.npy'):
        arr = np.load(file)
        if len(arr.shape) != 3:
            continue  # skip malformed files
        X.append(arr)
        # Example labeling: use filename keyword or simulated label
        label = 0 if 'poor' in file.name.lower() else 1
        y += [label] * arr.shape[0]
    X = np.concatenate(X)
    y = np.array(y)
    return torch.tensor(X).float(), torch.tensor(y).long()

X, y = load_data(preprocessed_path)
print(f"Loaded EEG segments: {X.shape}, Labels: {y.shape}")

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


Loaded EEG segments: torch.Size([3517, 19, 1280]), Labels: torch.Size([3517])


## Define Baseline models

In [4]:
import torch.nn as nn

class EEGNet(nn.Module):
    def __init__(self, n_channels=19, n_classes=2):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, (1, 64), padding=(0, 32))
        self.bn1 = nn.BatchNorm2d(8)
        self.conv2 = nn.Conv2d(8, 16, (n_channels, 1))
        self.bn2 = nn.BatchNorm2d(16)
        self.fc = nn.Linear(16, n_classes)

    def forward(self, x):
        x = torch.relu(self.bn1(self.conv1(x)))
        x = torch.relu(self.bn2(self.conv2(x)))
        x = torch.mean(x, dim=(2, 3))  # Global average pooling
        return self.fc(x)

class MLP(nn.Module):
    def __init__(self, input_size, hidden=128, n_classes=2):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_size, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_classes)
        )
    def forward(self, x):
        return self.fc(x)


## Initialize Model

In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

if model_type == 'EEGNet':
    model = EEGNet().to(device)
else:
    input_size = X.shape[1] * X.shape[2]
    model = MLP(input_size).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()


## Training loop

In [6]:
for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)
        if model_type == 'EEGNet':
            Xb = Xb.unsqueeze(1)  # Add channel dimension
        else:
            Xb = Xb.view(Xb.size(0), -1)  # Flatten for MLP

        optimizer.zero_grad()
        preds = model(Xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss/len(loader):.4f}")


Epoch 1/10 - Loss: 0.2565
Epoch 2/10 - Loss: 0.0766
Epoch 3/10 - Loss: 0.0292
Epoch 4/10 - Loss: 0.0146
Epoch 5/10 - Loss: 0.0087
Epoch 6/10 - Loss: 0.0055
Epoch 7/10 - Loss: 0.0038
Epoch 8/10 - Loss: 0.0028
Epoch 9/10 - Loss: 0.0021
Epoch 10/10 - Loss: 0.0017


## Evaluate

In [7]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)
        if model_type == 'EEGNet':
            Xb = Xb.unsqueeze(1)
        else:
            Xb = Xb.view(Xb.size(0), -1)
        preds = model(Xb)
        all_preds.extend(preds.argmax(1).cpu().numpy())
        all_true.extend(yb.cpu().numpy())

acc = accuracy_score(all_true, all_preds)
f1 = f1_score(all_true, all_preds)
print(f"✅ Accuracy: {acc:.3f} | F1: {f1:.3f}")


✅ Accuracy: 1.000 | F1: 1.000


✅ **Output:** Baseline model metrics (accuracy and F1-score).  
Use these as your reference when training EEG-BERT or multimodal fusion models.
